In [11]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

from pathlib import Path

# Primary directory for medical reports
REPORTS_DIR = Path(
    r"D:\TECHROVER\PROJECTS\RTH\healh-vault-project\Medical Reports"
)

# Document selection
# Use the same PDF that was analyzed in the previous OCR run.
SELECTED_DOCUMENTS = [] #["body_scan_report.pdf"]

# Preferred summary language for Stage 8
# Supported:
# "english", "gujarati", "hindi", "marathi", "tamil"
PREFERRED_LANGUAGE = "english"

# Storage bucket override
# None uses container.settings.patient_documents_bucket
BENCHMARK_BUCKET_OVERRIDE = None

print("Configuration Loaded:")
print(f"  Reports Directory:  {REPORTS_DIR}")
print(
    f"  Selected Documents: "
    f"{SELECTED_DOCUMENTS if SELECTED_DOCUMENTS else 'ALL'}"
)


Configuration Loaded:
  Reports Directory:  D:\TECHROVER\PROJECTS\RTH\healh-vault-project\Medical Reports
  Selected Documents: ALL


In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\TECHROVER\health-vault\health-vault-backend\ai-service"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Project exists: {PROJECT_ROOT.exists()}")

Project root: D:\TECHROVER\health-vault\health-vault-backend\ai-service
Project exists: True


In [13]:
import time
from uuid import uuid4

from app.settings import Settings
from app.container import Container

# ==============================================================================
# Initialize application
# ==============================================================================

settings = Settings()
container = Container(settings)

# ==============================================================================
# Determine documents to process
# ==============================================================================

if SELECTED_DOCUMENTS:
    documents = SELECTED_DOCUMENTS
else:
    documents = [
        file.name
        for file in REPORTS_DIR.iterdir()
        if file.is_file()
    ]

print("=" * 80)
print(f"DOCUMENTS TO PROCESS: {len(documents)}")
print("=" * 80)

raw_ocr_results = {}

# Store timing for every document
document_timings = {}

# Total benchmark timer
total_start_time = time.perf_counter()

# ==============================================================================
# Process documents
# ==============================================================================

for index, document_name in enumerate(documents, start=1):

    FILE_PATH = REPORTS_DIR / document_name

    print()
    print("=" * 80)
    print(f"[{index}/{len(documents)}] Processing: {document_name}")
    print("=" * 80)

    if not FILE_PATH.exists():

        print(f"ERROR: File not found: {FILE_PATH}")

        raw_ocr_results[document_name] = {
            "status": "failed",
            "error": "File not found",
        }

        document_timings[document_name] = {
            "status": "failed",
            "elapsed_seconds": 0,
        }

        continue

    # --------------------------------------------------------------------------
    # Start document timer
    # --------------------------------------------------------------------------

    document_start_time = time.perf_counter()

    try:

        raw_ocr_data = await container.ocr_stage_handler.run_ocr(
            file_path=FILE_PATH,
            filename=FILE_PATH.name,
            job_id=uuid4(),
            file_key=FILE_PATH.stem,
            current_pct=0,
            completed_stages=[],
            checkpoint_data={},
        )

        # ----------------------------------------------------------------------
        # Stop document timer
        # ----------------------------------------------------------------------

        document_elapsed = time.perf_counter() - document_start_time

        document_timings[document_name] = {
            "status": "success",
            "elapsed_seconds": document_elapsed,
        }

        raw_ocr_results[document_name] = {
            "status": "success",
            "data": raw_ocr_data,
            "elapsed_seconds": document_elapsed,
        }

        print(f"SUCCESS: {document_name}")
        print(
            f"Processing time: "
            f"{document_elapsed:.2f} seconds "
            f"({document_elapsed / 60:.2f} minutes)"
        )

        print("-" * 80)
        print("OCR RESULT")
        print("-" * 80)
        print(raw_ocr_data)

    except Exception as exc:

        document_elapsed = time.perf_counter() - document_start_time

        document_timings[document_name] = {
            "status": "failed",
            "elapsed_seconds": document_elapsed,
            "error": str(exc),
        }

        raw_ocr_results[document_name] = {
            "status": "failed",
            "error": str(exc),
            "elapsed_seconds": document_elapsed,
        }

        print(f"FAILED: {document_name}")
        print(f"Error: {exc}")
        print(
            f"Time before failure: "
            f"{document_elapsed:.2f} seconds "
            f"({document_elapsed / 60:.2f} minutes)"
        )

# ==============================================================================
# Total processing time
# ==============================================================================

total_elapsed = time.perf_counter() - total_start_time

successful = sum(
    1
    for result in raw_ocr_results.values()
    if result["status"] == "success"
)

failed = len(raw_ocr_results) - successful

# ==============================================================================
# Final summary
# ==============================================================================

print()
print("=" * 80)
print("OCR PROCESSING COMPLETE")
print("=" * 80)

print(f"Total documents:      {len(raw_ocr_results)}")
print(f"Successful:           {successful}")
print(f"Failed:               {failed}")
print(
    f"Total processing:     {total_elapsed:.2f} seconds "
    f"({total_elapsed / 60:.2f} minutes)"
)

print()
print("=" * 80)
print("DOCUMENT TIMING SUMMARY")
print("=" * 80)

for document_name, timing in document_timings.items():

    seconds = timing["elapsed_seconds"]

    print(
        f"{document_name:<40} "
        f"{seconds:>10.2f} sec "
        f"({seconds / 60:>7.2f} min) "
        f"[{timing['status']}]"
    )

DOCUMENTS TO PROCESS: 41

[1/41] Processing: 5069200904.pdf


ollama_vision_transport_error
ollama_vision_transport_error
ollama_vision_transport_error
ollama_vision_transport_error
ai_request_error
Vision OCR failed on page 1: AI vision request failed: AI circuit breaker is open for 28.5s after repeated failures
ai_request_error
Vision OCR failed on page 4: AI vision request failed: AI circuit breaker is open for 28.5s after repeated failures
ai_request_error
Vision OCR failed on page 5: AI vision request failed: AI circuit breaker is open for 28.5s after repeated failures
ai_request_error
Vision OCR failed on page 3: AI vision request failed: AI circuit breaker is open for 28.5s after repeated failures
ai_request_error
Vision OCR failed on page 2: AI vision request failed: AI circuit breaker is open for 26.8s after repeated failures
ai_request_error
Vision OCR failed on page 6: AI vision request failed: AI circuit breaker is open for 26.8s after repeated failures


SUCCESS: 5069200904.pdf
Processing time: 26.97 seconds (0.45 minutes)
--------------------------------------------------------------------------------
OCR RESULT
--------------------------------------------------------------------------------
{'pages': [{'page': 1, 'text': '', 'confidence': 0.0, 'lines': [], 'elapsed_ms': 22890, 'engine': 'none', 'status': 'FAILED', 'fallback_reason': 'primary_init_failed: PaddleOCR worker process is not initialized: Unknown argument: use_gpu', 'error': 'AI vision request failed: AI circuit breaker is open for 28.5s after repeated failures', 'cached': False, 'telemetry': {}, 'stage_timings': {'queue_wait_ms': 0.01, 'render_ms': 1757.73, 'preprocess_ms': 0.03, 'lang_detect_ms': 15.55, 'engine_init_ms': 0.02, 'det_ms': 0.0, 'rec_ms': 0.0, 'ocr_infer_ms': 0.0, 'quality_gate_ms': 0.0, 'vlm_call_ms': 0.0, 'postprocess_ms': 0.0, 'total_page_ms': 22890.0}}, {'page': 2, 'text': '', 'confidence': 0.0, 'lines': [], 'elapsed_ms': 1617, 'engine': 'none', 'status':

ai_request_error
Vision OCR failed on page 3: AI vision request failed: AI circuit breaker is open for 24.5s after repeated failures
ai_request_error
Vision OCR failed on page 4: AI vision request failed: AI circuit breaker is open for 24.5s after repeated failures


SUCCESS: 60300505154.pdf
Processing time: 2.19 seconds (0.04 minutes)
--------------------------------------------------------------------------------
OCR RESULT
--------------------------------------------------------------------------------
{'pages': [{'page': 1, 'text': 'RBC Paramters\nHemoglobin:\nL\ng/dL\n13.0 - 17.0\n11.4\nRBC Count\nL\nMil./cu mm\n4.5 - 5.5\n3.82\nHematocrit (calculated)\nL\n%\n40 - 50\n33.8\nM.C.V.\nfemto L\n83 - 101\n88.5\nM.C.H.\npico gms\n27 - 32\n29.9\nMCHC\ngm/L\n31.5 - 34.5\n33.8\nRDW\nH\n%\n11.5-14.5\n14.60\nMCV/RBC Ratio\n> 13\n23\nNucleated RBCs\n/100 WBCs\n0\nWBC Parameters\nTotal WBC Count\ncells/cumm\n4000 - 10000\n7880\nNeutrophils\n%\n38 - 70\n69.6\nLymphocytes\nL\n%\n21 - 49\n20.9\nEosinophils\n%\n0 - 7\n2.3\nMonocytes\n%\n3 - 11\n6.6\nBasophils\n%\n0 - 2\n0.6\nAbsolute Neutrophil Count\n/cmm\n1800 - 7700\n5484\nAbsolute Lymphocyte Count\n/cmm\n1000 - 3900\n1647\nAbsolute Eosinophil Count\n/cmm\n20 - 500\n181\nNLR\n3.3\nPlatelet Parameters\nPlate

...

SUCCESS: Media (18).jpg
Processing time: 1.70 seconds (0.03 minutes)
--------------------------------------------------------------------------------
OCR RESULT
--------------------------------------------------------------------------------
{'pages': [{'page': 1, 'text': '', 'confidence': 0.0, 'lines': [], 'elapsed_ms': 1695, 'engine': 'none', 'status': 'FAILED', 'fallback_reason': 'primary_init_failed: PaddleOCR worker process is not initialized: Unknown argument: use_gpu', 'error': 'AI vision request failed: AI circuit breaker is open for 2.2s after repeated failures', 'cached': False, 'telemetry': {}, 'stage_timings': {'queue_wait_ms': 0.0, 'render_ms': 0.0, 'preprocess_ms': 0.04, 'lang_detect_ms': 0.03, 'engine_init_ms': 0.02, 'det_ms': 0.0, 'rec_ms': 0.0, 'ocr_infer_ms': 0.0, 'quality_gate_ms': 0.0, 'vlm_call_ms': 0.0, 'postprocess_ms': 0.0, 'total_page_ms': 1695.0}}], 'fullText': '[PAGE 1 OCR FAILED - CLINICAL DATA UNREADABLE]', 'text': '[PAGE 1 OCR FAILED - CLINICAL DATA UNREAD

ai_request_error
Vision OCR failed on page 1: AI vision request failed: AI circuit breaker is open for 0.3s after repeated failures


SUCCESS: Media (19).jpg
Processing time: 1.92 seconds (0.03 minutes)
--------------------------------------------------------------------------------
OCR RESULT
--------------------------------------------------------------------------------
{'pages': [{'page': 1, 'text': '', 'confidence': 0.0, 'lines': [], 'elapsed_ms': 1865, 'engine': 'none', 'status': 'FAILED', 'fallback_reason': 'primary_init_failed: PaddleOCR worker process is not initialized: Unknown argument: use_gpu', 'error': 'AI vision request failed: AI circuit breaker is open for 0.3s after repeated failures', 'cached': False, 'telemetry': {}, 'stage_timings': {'queue_wait_ms': 0.0, 'render_ms': 0.0, 'preprocess_ms': 0.05, 'lang_detect_ms': 0.04, 'engine_init_ms': 0.03, 'det_ms': 0.0, 'rec_ms': 0.0, 'ocr_infer_ms': 0.0, 'quality_gate_ms': 0.0, 'vlm_call_ms': 0.0, 'postprocess_ms': 0.0, 'total_page_ms': 1865.0}}], 'fullText': '[PAGE 1 OCR FAILED - CLINICAL DATA UNREADABLE]', 'text': '[PAGE 1 OCR FAILED - CLINICAL DATA UNREAD

ollama_vision_http_error
ollama_vision_http_error
ollama_vision_http_error
ai_request_error
Vision OCR failed on page 14: AI vision request failed: Ollama vision request failed with HTTP 500: {"error":"prediction aborted, token repeat limit reached"}


SUCCESS: P1(CIMS)(27112023).pdf
Processing time: 309.90 seconds (5.17 minutes)
--------------------------------------------------------------------------------
OCR RESULT
--------------------------------------------------------------------------------
{'pages': [{'page': 1, 'text': 'Got it, let\'s transcribe the document exactly as it appears. First, check the header: "cims.org" at the top left, then "MarengoCIMS Hospital" with the logo. Then the table for Non Invasive Cardiology.\nStart with the table:\nUHID / IP NO: 300206730 (217842) | RISNo. | 5488027\nPatient Name: Mrs. MANJULABEN VALJIBHAI RANOLIYA | Age/Gender: 69 Y/F\nAdmitting Doctor: | Treating Doctor: Dr. Ajay Madhukar NaiK\nCath No: | Date of Procedure: 10/07/2024 12.22PM\nProcedure Code: | Echo Screening For LV Function\nWait, the table columns: first column is UHID/IP NO, Patient Name, Admitting Doctor, Cath No, Procedure Code. Second column is RISNo., Age/Gender, Treating Doctor, Date of Procedure. Wait, no, the table st